# 09 — Kaggle Competition Playbook (Lesson)

## Learning Objectives
- Understand **why** each mathematical term appears and how it changes optimization/generalization.
- Apply the technique to both classification and regression.
- Compare synthetic and real-data behavior with reproducible experiments.

## Driving Question
> How do we convert notebook experiments into robust leaderboard strategy?

## Notebook Roadmap
Math (LaTeX) -> Synthetic Data -> Real Data (MNIST/California Housing) -> Visualizations -> Best Practices -> Exercises

### Mathematical Foundation (First Principles)

Competition optimization should target expected private leaderboard performance:

$$
\theta^* = \arg\min_{\theta \in \Theta} \mathbb{E}[\mathcal{M}_{\text{private}}(\theta)]
$$

Cross-validation provides an estimator:

$$
\widehat{\mathcal{M}}_{\text{CV}}(\theta)=\frac{1}{K}\sum_{k=1}^K \mathcal{M}^{(k)}(\theta)
$$

Model averaging for two predictors:

$$
\hat{y}_{\text{ens}} = w\hat{y}_1 + (1-w)\hat{y}_2,\quad w\in[0,1]
$$

The practical challenge is minimizing **public-private shake** while iterating quickly.

### Deep Equation Lineage (Term-by-Term)

We now connect every lesson to the same first-principles optimization pipeline.

1. **Population objective** (what we really care about):
$$
\mathcal{R}(\theta)=\mathbb{E}_{(x,y)\sim\mathcal{D}}[\ell(f_\theta(x),y)]
$$
2. **Empirical approximation** (what we can compute):
$$
\hat{\mathcal{R}}(\theta)=\frac{1}{N}\sum_{i=1}^{N}\ell(f_\theta(x_i),y_i)
$$
3. **Mini-batch stochastic estimate** (what we optimize per step):
$$
g_t=\frac{1}{m}\sum_{i \in \mathcal{B}_t}\nabla_\theta \ell(f_\theta(x_i),y_i)
$$
4. **Chain-rule expansion** (how gradients flow through layers):
$$
\nabla_\theta \ell
=
\frac{\partial \ell}{\partial \hat{y}}
\cdot
\frac{\partial \hat{y}}{\partial h_L}
\cdot
\frac{\partial h_L}{\partial h_{L-1}}
\cdots
\frac{\partial h_1}{\partial \theta}
$$
5. **Regularized update** (how each lesson perturbs the step):
$$
\theta_{t+1}
=
\theta_t
-\eta\left(g_t+\nabla_\theta\Omega(\theta_t)\right)
$$

| Term | Meaning | Where it appears in code |
|---|---|---|
| $\eta$ | step size controlling update magnitude | optimizer learning rate |
| $g_t$ | stochastic gradient estimate | `loss.backward()` |
| $\Omega(\theta)$ | explicit/implicit regularization | weight decay, dropout, early stopping |
| $m$ | mini-batch size | `DataLoader(..., batch_size=...)` |
| $\hat{y}=f_\theta(x)$ | model predictions | `logits = model(xb)` |

**Interpretation checkpoint:** if training is unstable, inspect whether the issue comes from gradient scale ($g_t$), step size ($\eta$), or noisy estimation (small $m$).

### Before the Code: Purpose + Mechanics

This setup cell builds a reproducible tabular-competition environment with optional XGBoost/LightGBM support.
Background: robust benchmarking requires stable folds, consistent preprocessing, and graceful fallbacks when some libraries are unavailable.

In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_california_housing, make_classification
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, mean_squared_error, roc_curve
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

# Optional libraries: if unavailable, notebook still runs with sklearn fallbacks.
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


def make_ohe():
    # Compatibility helper for older/newer sklearn versions.
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def build_tabular_frame():
    # Use California Housing and add competition-style categorical/missingness structure.
    raw = fetch_california_housing(as_frame=True)
    X = raw.frame.drop(columns=['MedHouseVal']).copy()
    y = raw.frame['MedHouseVal'].astype(float).copy()

    X['income_bucket'] = pd.qcut(X['MedInc'], q=5, labels=False, duplicates='drop').astype(str)
    X['house_age_bucket'] = pd.cut(X['HouseAge'], bins=[0, 15, 30, 45, 60], include_lowest=True).astype(str)
    X['geo_bucket'] = (X['Latitude'].round(1).astype(str) + '_' + X['Longitude'].round(1).astype(str))

    # Inject mild missingness to mimic real competition cleanup requirements.
    missing_mask = np.random.rand(len(X)) < 0.03
    X.loc[missing_mask, 'AveRooms'] = np.nan
    X.loc[missing_mask, 'AveBedrms'] = np.nan
    return X, y


class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout_p=0.15, output_dim=1):
        super().__init__()
        layers = []
        prev = input_dim
        for hidden in hidden_dims:
            layers.append(nn.Linear(prev, hidden))
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev = hidden
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def make_tabular_preprocessor(num_cols, cat_cols):
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), num_cols),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('ohe', make_ohe())
            ]), cat_cols),
        ]
    )


def fit_preprocessor_matrices(preprocessor, X_train, X_valid):
    X_train_m = preprocessor.fit_transform(X_train)
    X_valid_m = preprocessor.transform(X_valid)
    if hasattr(X_train_m, 'toarray'):
        X_train_m = X_train_m.toarray()
    if hasattr(X_valid_m, 'toarray'):
        X_valid_m = X_valid_m.toarray()
    return np.asarray(X_train_m, dtype=np.float32), np.asarray(X_valid_m, dtype=np.float32)


def make_tabular_loaders(X_train_np, y_train, X_valid_np, y_valid, batch_size=256, task='regression'):
    y_train_t = torch.tensor(np.asarray(y_train, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    y_valid_t = torch.tensor(np.asarray(y_valid, dtype=np.float32).reshape(-1, 1), dtype=torch.float32)
    train_ds = TensorDataset(torch.tensor(X_train_np, dtype=torch.float32), y_train_t)
    valid_ds = TensorDataset(torch.tensor(X_valid_np, dtype=torch.float32), y_valid_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False)
    return train_loader, valid_loader


def train_tabular_mlp(
    X_train,
    y_train,
    X_valid,
    y_valid,
    task='regression',
    hidden_dims=(256, 128),
    dropout_p=0.15,
    lr=8e-4,
    weight_decay=1e-5,
    batch_size=256,
    epochs=40,
    patience=6,
    verbose=False,
):
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]
    preprocessor = make_tabular_preprocessor(num_cols, cat_cols)
    X_train_np, X_valid_np = fit_preprocessor_matrices(preprocessor, X_train, X_valid)
    train_loader, valid_loader = make_tabular_loaders(
        X_train_np, y_train, X_valid_np, y_valid, batch_size=batch_size, task=task
    )

    model = TabularMLP(
        input_dim=X_train_np.shape[1],
        hidden_dims=hidden_dims,
        dropout_p=dropout_p,
        output_dim=1,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss() if task == 'classification' else nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}
    best_state = copy.deepcopy(model.state_dict())
    best_val = float('inf')
    best_epoch = 1
    wait = 0
    stopped_epoch = epochs

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                val_losses.append(criterion(out, yb).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if verbose:
            print(f'Epoch {epoch+1:02d}: train={train_loss:.5f} val={val_loss:.5f}')

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            stopped_epoch = epoch + 1
            break

    model.load_state_dict(best_state)

    with torch.no_grad():
        valid_tensor = torch.tensor(X_valid_np, dtype=torch.float32).to(device)
        valid_logits = model(valid_tensor).squeeze(1).cpu().numpy()

    if task == 'classification':
        valid_pred = 1.0 / (1.0 + np.exp(-valid_logits))
    else:
        valid_pred = valid_logits

    return {
        'model': model,
        'preprocessor': preprocessor,
        'history': history,
        'val_pred': valid_pred,
        'best_epoch': best_epoch,
        'stopped_epoch': stopped_epoch,
    }


def predict_tabular_mlp(bundle, X_df, task='regression'):
    X_np = bundle['preprocessor'].transform(X_df)
    if hasattr(X_np, 'toarray'):
        X_np = X_np.toarray()
    X_tensor = torch.tensor(np.asarray(X_np, dtype=np.float32), dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = bundle['model'](X_tensor).squeeze(1).cpu().numpy()
    if task == 'classification':
        return 1.0 / (1.0 + np.exp(-logits))
    return logits


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


TABULAR_MLP_BASE_CFG = {
    'hidden_dims': (256, 128),
    'dropout_p': 0.15,
    'lr': 8e-4,
    'weight_decay': 1e-5,
    'batch_size': 256,
    'epochs': 40,
    'patience': 6,
}


def run_tabular_mlp_cv(X_df, y, cfg, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_pred = np.zeros(len(X_df), dtype=np.float32)
    fold_scores = []

    for fold_idx, (tr_idx, va_idx) in enumerate(kf.split(X_df), start=1):
        X_tr, X_va = X_df.iloc[tr_idx], X_df.iloc[va_idx]
        y_tr, y_va = np.asarray(y)[tr_idx], np.asarray(y)[va_idx]
        bundle = train_tabular_mlp(X_tr, y_tr, X_va, y_va, task='regression', **cfg)
        pred = bundle['val_pred']
        score = rmse(y_va, pred)
        fold_scores.append(score)
        oof_pred[va_idx] = pred
        print(f'Fold {fold_idx}: RMSE={score:.5f}, best_epoch={bundle["best_epoch"]}, stop_epoch={bundle["stopped_epoch"]}')

    return {
        'fold_scores': fold_scores,
        'oof_pred': oof_pred,
        'rmse_mean': float(np.mean(fold_scores)),
        'rmse_std': float(np.std(fold_scores)),
    }

### After the Code: Background + Why It Can Help

This setup emphasizes reproducibility and fairness: same folds, same preprocessing skeleton, and explicit fallback behavior.
For lessons 08 and 09 we now share one PyTorch MLP pipeline (preprocessing -> loaders -> training loop -> inference) so modeling choices stay aligned.

## Kaggle Playbook Simulation
We simulate a public/private leaderboard split and train a repeatable decision workflow around CV, shake risk, and ensembling.

### Before the Code: Purpose + Mechanics

This cell builds a competition-style evaluation stack with a hidden private split.
Background: robust Kaggle prep means optimizing validation quality and shake resistance, not only public leaderboard bumps.

In [ ]:
X_raw, y = build_tabular_frame()
X = X_raw.copy()
X['rooms_per_household'] = X['AveRooms'] / (X['AveOccup'] + 1e-6)
X['income_x_occupancy'] = X['MedInc'] * X['AveOccup']

X_train_full, X_lb, y_train_full, y_lb = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_public, X_private, y_public, y_private = train_test_split(X_lb, y_lb, test_size=0.50, random_state=SEED)

# Two PyTorch MLP candidates with different capacity/regularization choices.
candidates = {
    'mlp_deep': {**TABULAR_MLP_BASE_CFG, 'hidden_dims': (320, 160, 64), 'dropout_p': 0.12, 'epochs': 45, 'patience': 7},
    'mlp_regularized': {**TABULAR_MLP_BASE_CFG, 'hidden_dims': (192, 96), 'dropout_p': 0.24, 'weight_decay': 3e-5, 'epochs': 45, 'patience': 7},
}

rows = []
cv_fold_rows = []
trained_candidates = {}
public_pred_store = {}
private_pred_store = {}

for name, cfg_local in candidates.items():
    cv_out = run_tabular_mlp_cv(X_train_full, y_train_full, cfg_local, n_splits=5)
    for fold_idx, fold_score in enumerate(cv_out['fold_scores'], start=1):
        cv_fold_rows.append({'model': name, 'fold': fold_idx, 'rmse': fold_score})

    # Train submission-facing model and score public/private proxies.
    bundle = train_tabular_mlp(X_train_full, y_train_full, X_public, y_public, task='regression', **cfg_local)
    pred_public = bundle['val_pred']
    pred_private = predict_tabular_mlp(bundle, X_private, task='regression')

    trained_candidates[name] = bundle
    public_pred_store[name] = pred_public
    private_pred_store[name] = pred_private

    rows.append({
        'model': name,
        'cv_proxy_score': cv_out['rmse_mean'],
        'public_proxy_score': rmse(y_public, pred_public),
        'private_proxy_score': rmse(y_private, pred_private),
        'best_epoch': bundle['best_epoch'],
        'stop_epoch': bundle['stopped_epoch']
    })

leaderboard = pd.DataFrame(rows)
leaderboard['shake_abs'] = (leaderboard['private_proxy_score'] - leaderboard['public_proxy_score']).abs()
cv_fold_df = pd.DataFrame(cv_fold_rows)
display(leaderboard.sort_values('private_proxy_score'))

### After the Code: Background + Why It Can Help

Treat this as a process rehearsal: build habits for split design, leakage control, and shake-aware model selection.
A model with slightly worse public score but lower shake can be the safer final submission.

### Before the Code: Purpose + Mechanics

This diagnostics block mirrors an early Kaggle EDA pass: check split shift, missingness structure, and feature-target behavior before tuning.
Background: many leaderboard failures are data-split problems disguised as model problems.

In [ ]:
key_features = ['MedInc', 'HouseAge', 'AveRooms', 'Population']
shift_df = pd.concat([
    X_train_full[key_features].assign(split='train_full'),
    X_public[key_features].assign(split='public'),
    X_private[key_features].assign(split='private')
], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), key_features):
    sns.kdeplot(data=shift_df, x=col, hue='split', fill=True, alpha=0.2, common_norm=False, ax=ax)
    ax.set_title(f'Distribution shift check: {col}')
plt.tight_layout()
plt.show()

missing_by_split = pd.DataFrame({
    'train_full': X_train_full.isna().mean(),
    'public': X_public.isna().mean(),
    'private': X_private.isna().mean()
}) * 100
top_missing_cols = missing_by_split.max(axis=1).sort_values(ascending=False).head(10).index

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.heatmap(missing_by_split.loc[top_missing_cols], annot=True, fmt='.2f', cmap='Reds', ax=axes[0])
axes[0].set_title('Missingness heatmap (% by split)')

missing_plot = missing_by_split.loc[top_missing_cols].reset_index().melt(id_vars='index', var_name='split', value_name='missing_pct')
missing_plot = missing_plot.rename(columns={'index': 'feature'})
sns.barplot(data=missing_plot, x='missing_pct', y='feature', hue='split', ax=axes[1])
axes[1].set_title('Top missingness bars by split')
axes[1].set_xlabel('Missing %')
plt.tight_layout()
plt.show()

target_rel = pd.concat([
    X_train_full[['MedInc', 'AveRooms']].assign(target=y_train_full.values, split='train_full'),
    X_public[['MedInc', 'AveRooms']].assign(target=y_public.values, split='public'),
    X_private[['MedInc', 'AveRooms']].assign(target=y_private.values, split='private')
], ignore_index=True)

g = sns.lmplot(
    data=target_rel.sample(min(3000, len(target_rel)), random_state=SEED),
    x='MedInc',
    y='target',
    hue='split',
    scatter_kws={'alpha': 0.25, 's': 12},
    height=4,
    aspect=1.5
)
g.fig.suptitle('Feature-target relationship by split (MedInc vs target)', y=1.02)
plt.show()

### After the Code: Background + Why It Can Help

If one split has visibly shifted feature distribution or broken feature-target slope, prioritize validation redesign before heavy model search.
This step often explains public/private shake more than any hyperparameter tweak.

### Before the Code: Purpose + Mechanics

This solved strategy cell evaluates simple ensembling and run-card logging.
Background: Kaggle gains often come from disciplined small improvements plus strong experiment bookkeeping.

In [ ]:
weights = np.linspace(0.0, 1.0, 11)
ens_rows = []
for w in weights:
    ens_public = w * public_pred_store['mlp_deep'] + (1 - w) * public_pred_store['mlp_regularized']
    ens_private = w * private_pred_store['mlp_deep'] + (1 - w) * private_pred_store['mlp_regularized']
    ens_rows.append({
        'weight_mlp_deep': float(w),
        'public_rmse': rmse(y_public, ens_public),
        'private_rmse': rmse(y_private, ens_private)
    })

ens_df = pd.DataFrame(ens_rows)
ens_df['shake_abs'] = (ens_df['private_rmse'] - ens_df['public_rmse']).abs()
display(ens_df.sort_values('public_rmse').head(6))

best_idx = ens_df['private_rmse'].idxmin()
print('Most private-robust ensemble row:')
display(ens_df.loc[[best_idx]])

run_card = {
    'seed': SEED,
    'folds': 5,
    'feature_version': 'v1_rooms_income_interactions',
    'model_family': 'pytorch_tabular_mlp_blend',
    'notes': 'Selected by private-robust simulated leaderboard, not just public score'
}
print('Run card:', run_card)

### After the Code: Background + Why It Can Help

Use this template to avoid ad-hoc submissions: log assumptions, compare robustly, and submit with an explicit risk rationale.
Kaggle consistency usually beats one-off lucky spikes.

### Before the Code: Purpose + Mechanics

This final competition dashboard compares fold stability and leaderboard-style behavior across candidates and blends.
Background: deciding what to submit is mostly a visualization + risk-management problem.

In [ ]:
leaderboard_plot = leaderboard.copy()
leaderboard_plot['leaderboard_gap'] = leaderboard_plot['private_proxy_score'] - leaderboard_plot['public_proxy_score']

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.boxplot(data=cv_fold_df, x='model', y='rmse', ax=axes[0], palette='Set3')
sns.stripplot(data=cv_fold_df, x='model', y='rmse', ax=axes[0], color='black', size=4, alpha=0.6)
axes[0].set_title('CV-fold RMSE comparison (PyTorch MLP)')

sns.scatterplot(data=leaderboard_plot, x='public_proxy_score', y='private_proxy_score', hue='model', s=130, ax=axes[1])
for _, row in leaderboard_plot.iterrows():
    axes[1].annotate(row['model'], (row['public_proxy_score'], row['private_proxy_score']), xytext=(4, 4), textcoords='offset points')
axes[1].set_title('Leaderboard-style public vs private proxy')
axes[1].plot(
    [leaderboard_plot['public_proxy_score'].min(), leaderboard_plot['public_proxy_score'].max()],
    [leaderboard_plot['public_proxy_score'].min(), leaderboard_plot['public_proxy_score'].max()],
    linestyle='--', color='gray', linewidth=1
)

sns.lineplot(data=ens_df, x='weight_mlp_deep', y='public_rmse', marker='o', label='public_rmse', ax=axes[2])
sns.lineplot(data=ens_df, x='weight_mlp_deep', y='private_rmse', marker='o', label='private_rmse', ax=axes[2])
axes[2].set_title('Blend weight leaderboard trace')
axes[2].set_ylabel('RMSE')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=leaderboard_plot.sort_values('shake_abs', ascending=False), x='model', y='shake_abs', palette='rocket')
plt.title('Shake-risk leaderboard (absolute public-private gap)')
plt.ylabel('Absolute gap')
plt.tight_layout()
plt.show()

### After the Code: Background + Why It Can Help

Ship candidates that are strong on both fold stability and shake-risk visuals, not just one metric snapshot.
These plots make submission decisions auditable and defensible under deadline pressure.

## Best Practices
- Treat feature engineering, CV design, and ensembling as a coupled system.
- Optimize for robust validation and shake resistance, not public leaderboard spikes.
- Log every run with data version, folds, seed, model hash, and metric deltas.

## Common Pitfalls
- Applying a technique without a baseline comparison.
- Drawing conclusions from training loss only.
- Ignoring seed sensitivity and run-to-run variance.

## Explanatory Depth Checkpoints
- **Why this workflow?** Because leaderboard gains are fragile without leakage-safe validation and reproducible logs.
- **Key idea:** Strong experiments isolate one hypothesis at a time so score movement has a clear causal explanation.
- **Important pitfall:** A public-score spike can hide private overfitting; always compare fold variance and shake risk.
- **In practice:** Keep assumptions explicit, test alternatives quickly, and record rollback plans before each submission.
- **Question:** Which diagnostic would make you reject a seemingly strong model?
- **Question:** How would you defend your final submission choice to a teammate under deadline pressure?

## Kaggle-Prep Exercise Bank (Dataset-Driven)
### Track A — Validation Design
1. **Exercise:** Build 5-fold, 10-fold, and GroupKFold variants; compare rank stability across folds.
2. **Exercise:** Create a leakage trap feature intentionally, detect it, and document your detection rule.
3. **Exercise:** Simulate temporal drift by sorting on a proxy time feature and using time-aware splits.

### Track B — Modeling Strategy
4. **Exercise:** Train at least three model families (linear, tree boosting, bagging) with a fixed budget.
5. **Exercise:** Build a score-vs-runtime frontier and justify your final Pareto-optimal candidates.
6. **Exercise:** Tune one model deeply and another model shallowly; compare marginal gain per minute.

### Track C — Ensembling and Risk
7. **Exercise:** Blend two models with weights from 0.0 to 1.0 and report public/private shake.
8. **Exercise:** Add a third model and test whether diversity beats raw single-model strength.
9. **Question:** Construct a submission policy: when should you trust CV over public leaderboard?

### Track D — Reporting and Communication
10. **Exercise:** Create an experiment registry with feature version, seed, fold schema, and commit hash.
11. **Exercise:** Write a post-mortem for one failed experiment and what diagnostic disproved the hypothesis.
12. **Exercise:** Draft a final competition report with reproducibility checklist and fallback plan.